In [103]:
import pandas as pd 

In [104]:
pd.set_option('display.max_columns', None)

## Before starting this phase, make sure to run the following three scripts in sequence. They work with the data obtained from Kaggle:

### 1. First Script: Data Preprocessing (gen_data.py)
This script processes the dataset by filtering the data based on a list of target artists from Europe and Japan. It then assigns regions (Europe or Japan) to the corresponding artists and saves the cleaned data into a new folder.

#### Key steps:
- Filters the dataset to only include tracks by specific European and Japanese artists.
- Assigns each track to a region (Europe or Japan) based on the artist.
- Saves the filtered data into separate CSV files.

---

### 2. Second Script: Data Alignment (merge_data.py)
This script combines multiple filtered datasets into one comprehensive dataset, aligning the columns with the predefined features. It ensures that all the data from the different files are merged properly, with consistent column names.

#### Key steps:
- Reads multiple CSV files from the filtered data directory.
- Aligns the columns of each dataset to a predefined set of features.
- Combines all the aligned datasets into a single file and saves it.

---

### 3. Third Script: Fetching Missing Data from Spotify API (filled_data.py)
This script fetches additional track and artist data from Spotify using the Spotify API. Be aware that the API has request limits, so it’s better to process the data in smaller batches.

#### Key steps:
- Uses the Spotify API to fill in missing data, such as genre, popularity, explicit status, and release date.
- Makes requests to the API for each track, handling potential errors gracefully.
- Saves the updated dataset with the additional information fetched from Spotify.

---

### Note:
If the data has already been preprocessed and cleaned, you can skip the data cleaning step and directly use the prepared dataset from the previous phases.

# **Prepeared data**

In [105]:
df = pd.read_csv('../data/combined_filled_data.csv')
df.head()

,Artist ID,Artist name,Region,Track ID,Track name,Release date,Genre,Popularity,Explicit,Duration,Acousticness,Danceability,Energy,Instrumentalness,Key,Liveness,Loudness,Mode,Speechiness,Tempo,Valence
0,7t0rwkOPGlDPEhaOcVtOt9,The Cranberries,Europe,0GrK2fO1EOBrH7WPseI32X,Zombie,2022-10-11,rock,1.0,0.0,307440,0.014500,0.252,0.738,0.000497,7,0.375,-5.326,1,0.0446,165.187,0.256
1,12Chz98pHFMPJEknJQMWvI,Muse,Europe,6kyxQuFD38mo4S3urD2Wkw,Unintended,1999-01-01,alternative rock,61.0,False,237027,0.647000,0.487,0.279,0.001850,6,0.110,-11.586,0,0.0265,139.362,0.162
2,4Z8W4fKeB5YxbusRsdQVPb,Radiohead,Europe,2sy0icOIskeP2lCqgZiTyE,Talk Show Host,1996-01-22,art rock,57.0,True,281000,0.277000,0.535,0.479,0.028500,F,0.139,-14.112,Minor,0.0311,88.841,0.504
3,24b231EnCM3BkpiuWs5VBE,SID,Japan,6LjEYqFZgktIxyF6s4Fc3d,ドラマ,2018-04-04,anime,0.0,0.0,229093,0.000034,0.457,0.950,0.002560,0,0.339,-3.235,1,0.0655,138.046,0.493
4,0MK8l3nURwwQIjafvXoJJt,ASIAN KUNG-FU GENERATION,Japan,3rxLb8cYhEnv3ZEoluDdpA,Re:Re: (Single ver.),2016-03-16,j-rock,55.0,0.0,332240,0.000249,0.464,0.868,0.000420,6,0.248,-5.051,0,0.0502,155.008,0.512


## **Check dataset**

In [106]:
df.dtypes

Artist ID            object
Artist name          object
Region               object
Track ID             object
Track name           object
Release date         object
Genre                object
Popularity          float64
Explicit             object
Duration              int64
Acousticness        float64
Danceability        float64
Energy              float64
Instrumentalness    float64
Key                  object
Liveness            float64
Loudness            float64
Mode                 object
Speechiness         float64
Tempo               float64
Valence             float64
dtype: object

In [107]:
df.isna().sum()

Artist ID            0
Artist name          0
Region               0
Track ID             0
Track name           0
Release date         0
Genre               60
Popularity           0
Explicit             0
Duration             0
Acousticness         0
Danceability         0
Energy               0
Instrumentalness     0
Key                  0
Liveness             0
Loudness             0
Mode                 0
Speechiness          0
Tempo                0
Valence              0
dtype: int64

In [108]:
df['Mode'].value_counts()

Mode
1        1246
Major     792
0         653
Minor     452
Name: count, dtype: int64

In [109]:
df['Key'].value_counts()

Key
7     273
9     266
2     257
0     230
4     184
A     166
D     164
11    156
G     149
5     143
C     120
1     117
B     109
E     103
F      94
C#     87
G#     87
6      82
F#     77
10     77
8      74
A#     60
3      40
D#     28
Name: count, dtype: int64

In [110]:
df['Explicit'].value_counts()

Explicit
False    1562
0.0      1522
1.0        33
True       26
Name: count, dtype: int64

In [111]:
df['Genre'].value_counts()

Genre
anime               638
j-rock              601
classic rock        541
art rock            353
progressive rock    245
indie               237
hard rock           139
alternative rock    118
indie rock          103
rock                 77
Dance                14
Rock                 12
Pop                   3
celtic                2
Name: count, dtype: int64

In [112]:
# there should be 24
df['Artist ID'].nunique()

26

## **Processing values**

### **Artist ID** feature

In [113]:
artist_counts = df.groupby(['Artist ID', 'Artist name']).size().reset_index(name='Track Count')
artist_counts_sorted = artist_counts.sort_values(by='Track Count', ascending=False)
artist_counts_sorted

,Artist ID,Artist name,Track Count
15,4Z8W4fKeB5YxbusRsdQVPb,Radiohead,353
5,0k17h0D3J5VfsdmQ1iZtE9,Pink Floyd,245
8,1dfeR4HaWDbWqFHLkxsg1d,Queen,242
23,7Ln80lUS6He07XvHI8qqHH,Arctic Monkeys,237
1,0MK8l3nURwwQIjafvXoJJt,ASIAN KUNG-FU GENERATION,237
11,36QJpDe2go2KgaRleHCDTp,Led Zeppelin,192
4,0blbVefuxOGltDBa00dspv,LiSA,179
3,0bAsR2unSRpn6BQPEnNlZm,Aimer,178
14,3w2HqkKa6upwuXEULtGvnY,FLOW,142
18,568ZhdwyaiCyOGJRtNYhWf,Deep Purple,139


In [114]:
artist_id_map = {
    '6rEzedK7cKWjeQWdAYvWVG': '1dfeR4HaWDbWqFHLkxsg1d',
    '6AnrSlk5Gp1YMXgaI3mWCL': '22bE4uQ6baNwSHPVcDxLCe'
}

df['Artist ID'] = df['Artist ID'].replace(artist_id_map)

### **Mode** feature

In [115]:
def mode_convert_num(df):
    df['Mode'] = df['Mode'].replace({'Major': 1, 'Minor': 0})
    df['Mode'] = pd.to_numeric(df['Mode'], errors='coerce').astype('Int64')
    return df

In [116]:
def mode_convert_string(df):
    df['Mode'] = df['Mode'].astype(str)
    df['Mode'] = df['Mode'].replace({'1': 'Major', '0': 'Minor'})
    return df

In [117]:
df = mode_convert_string(df)

### **Key** feature

In [118]:
def replace_with_key_map(df):
    key_map = {
        'C': 0, 'C#': 1, 'D': 2, 'D#': 3,
        'E': 4, 'F': 5, 'F#': 6, 'G': 7,
        'G#': 8, 'A': 9, 'A#': 10, 'B': 11
    }

    df['Key'] = df['Key'].replace(key_map)
    df['Key'] = pd.to_numeric(df['Key'], errors='coerce').astype('Int64')
    
    return df

In [119]:
def replace_with_inv_key_map(df):
    inv_key_map = {
        '0': 'C', '1': 'C#', '2': 'D', '3': 'D#',
        '4': 'E', '5': 'F', '6': 'F#', '7': 'G',
        '8': 'G#', '9': 'A', '10': 'A#', '11': 'B'
    }
    df['Key'] = df['Key'].astype(str)
    df['Key'] = df['Key'].replace(inv_key_map)

    return df

In [120]:
df = replace_with_inv_key_map(df)

### **Explicit** feature

In [121]:
df['Explicit'] = df['Explicit'].str.lower().map(lambda x: x in ['1', '1.0', 'true', 'yes'])

### **Release date** features

In [122]:
df['Release date'] = pd.to_datetime(df['Release date'], errors='coerce')

In [123]:
# or str: .astype(str)
df['Release date'] = df['Release date'].dt.to_period('M')

### **Genre** feature

In [124]:
df['Genre'] = df['Genre'].str.lower()

In [125]:
df['Genre'] = df.apply(lambda row: 'rock' if pd.isna(row['Genre']) and row['Region'] == 'Europe' 
                       else ('j-rock' if pd.isna(row['Genre']) and row['Region'] == 'Japan' 
                             else row['Genre']), axis=1)

### **Duration** feature

In [126]:
df['Duration'] = (df['Duration'] / 1000).round(2)

## **Save clean data**

In [127]:
df.dtypes

Artist ID              object
Artist name            object
Region                 object
Track ID               object
Track name             object
Release date        period[M]
Genre                  object
Popularity            float64
Explicit                 bool
Duration              float64
Acousticness          float64
Danceability          float64
Energy                float64
Instrumentalness      float64
Key                    object
Liveness              float64
Loudness              float64
Mode                   object
Speechiness           float64
Tempo                 float64
Valence               float64
dtype: object

In [128]:
df.isna().sum()

Artist ID           0
Artist name         0
Region              0
Track ID            0
Track name          0
Release date        0
Genre               0
Popularity          0
Explicit            0
Duration            0
Acousticness        0
Danceability        0
Energy              0
Instrumentalness    0
Key                 0
Liveness            0
Loudness            0
Mode                0
Speechiness         0
Tempo               0
Valence             0
dtype: int64

In [129]:
df.head()

,Artist ID,Artist name,Region,Track ID,Track name,Release date,Genre,Popularity,Explicit,Duration,Acousticness,Danceability,Energy,Instrumentalness,Key,Liveness,Loudness,Mode,Speechiness,Tempo,Valence
0,7t0rwkOPGlDPEhaOcVtOt9,The Cranberries,Europe,0GrK2fO1EOBrH7WPseI32X,Zombie,2022-10,rock,1.0,False,307.44,0.014500,0.252,0.738,0.000497,G,0.375,-5.326,Major,0.0446,165.187,0.256
1,12Chz98pHFMPJEknJQMWvI,Muse,Europe,6kyxQuFD38mo4S3urD2Wkw,Unintended,1999-01,alternative rock,61.0,False,237.03,0.647000,0.487,0.279,0.001850,F#,0.110,-11.586,Minor,0.0265,139.362,0.162
2,4Z8W4fKeB5YxbusRsdQVPb,Radiohead,Europe,2sy0icOIskeP2lCqgZiTyE,Talk Show Host,1996-01,art rock,57.0,True,281.00,0.277000,0.535,0.479,0.028500,F,0.139,-14.112,Minor,0.0311,88.841,0.504
3,24b231EnCM3BkpiuWs5VBE,SID,Japan,6LjEYqFZgktIxyF6s4Fc3d,ドラマ,2018-04,anime,0.0,False,229.09,0.000034,0.457,0.950,0.002560,C,0.339,-3.235,Major,0.0655,138.046,0.493
4,0MK8l3nURwwQIjafvXoJJt,ASIAN KUNG-FU GENERATION,Japan,3rxLb8cYhEnv3ZEoluDdpA,Re:Re: (Single ver.),2016-03,j-rock,55.0,False,332.24,0.000249,0.464,0.868,0.000420,F#,0.248,-5.051,Minor,0.0502,155.008,0.512


In [130]:
df.to_csv("../data/cleaned_data/spotify_europeVSjapan_rock.csv", index=False)

# **Prepeared data for analysis**

In [131]:
df = pd.read_csv('../data/cleaned_data/spotify_europeVSjapan_rock.csv')
df.head()

,Artist ID,Artist name,Region,Track ID,Track name,Release date,Genre,Popularity,Explicit,Duration,Acousticness,Danceability,Energy,Instrumentalness,Key,Liveness,Loudness,Mode,Speechiness,Tempo,Valence
0,7t0rwkOPGlDPEhaOcVtOt9,The Cranberries,Europe,0GrK2fO1EOBrH7WPseI32X,Zombie,2022-10,rock,1.0,False,307.44,0.014500,0.252,0.738,0.000497,G,0.375,-5.326,Major,0.0446,165.187,0.256
1,12Chz98pHFMPJEknJQMWvI,Muse,Europe,6kyxQuFD38mo4S3urD2Wkw,Unintended,1999-01,alternative rock,61.0,False,237.03,0.647000,0.487,0.279,0.001850,F#,0.110,-11.586,Minor,0.0265,139.362,0.162
2,4Z8W4fKeB5YxbusRsdQVPb,Radiohead,Europe,2sy0icOIskeP2lCqgZiTyE,Talk Show Host,1996-01,art rock,57.0,True,281.00,0.277000,0.535,0.479,0.028500,F,0.139,-14.112,Minor,0.0311,88.841,0.504
3,24b231EnCM3BkpiuWs5VBE,SID,Japan,6LjEYqFZgktIxyF6s4Fc3d,ドラマ,2018-04,anime,0.0,False,229.09,0.000034,0.457,0.950,0.002560,C,0.339,-3.235,Major,0.0655,138.046,0.493
4,0MK8l3nURwwQIjafvXoJJt,ASIAN KUNG-FU GENERATION,Japan,3rxLb8cYhEnv3ZEoluDdpA,Re:Re: (Single ver.),2016-03,j-rock,55.0,False,332.24,0.000249,0.464,0.868,0.000420,F#,0.248,-5.051,Minor,0.0502,155.008,0.512


In [132]:
df.shape

(3143, 21)

In [133]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3143 entries, 0 to 3142
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Artist ID         3143 non-null   object 
 1   Artist name       3143 non-null   object 
 2   Region            3143 non-null   object 
 3   Track ID          3143 non-null   object 
 4   Track name        3143 non-null   object 
 5   Release date      3143 non-null   object 
 6   Genre             3143 non-null   object 
 7   Popularity        3143 non-null   float64
 8   Explicit          3143 non-null   bool   
 9   Duration          3143 non-null   float64
 10  Acousticness      3143 non-null   float64
 11  Danceability      3143 non-null   float64
 12  Energy            3143 non-null   float64
 13  Instrumentalness  3143 non-null   float64
 14  Key               3143 non-null   object 
 15  Liveness          3143 non-null   float64
 16  Loudness          3143 non-null   float64


In [134]:
df.groupby("Artist name")["Popularity"].max()

Artist name
ASIAN KUNG-FU GENERATION    64.0
Aimer                       77.0
Arctic Monkeys              92.0
Deep Purple                 75.0
FLOW                        67.0
Franz Ferdinand             79.0
KANA-BOON                   73.0
Led Zeppelin                82.0
LiSA                        74.0
Ling tosite sigure          51.0
MY FIRST STORY              55.0
Muse                        83.0
ONE OK ROCK                 63.0
Pink Floyd                  77.0
Queen                       84.0
Radiohead                   91.0
SID                         60.0
Sayuri                      67.0
THE ORAL CIGARETTES         65.0
The Cranberries             82.0
The Police                  83.0
The Rolling Stones          80.0
U2                          79.0
WagakkiBand                 45.0
Name: Popularity, dtype: float64

In [135]:
df_sorted = df.sort_values(by=["Artist name", "Popularity"], ascending=[True, False])
df_top_15 = df_sorted.groupby("Artist name").head(15)
df_top_15

,Artist ID,Artist name,Region,Track ID,Track name,Release date,Genre,Popularity,Explicit,Duration,Acousticness,Danceability,Energy,Instrumentalness,Key,Liveness,Loudness,Mode,Speechiness,Tempo,Valence
393,0MK8l3nURwwQIjafvXoJJt,ASIAN KUNG-FU GENERATION,Japan,5ORPYXJKlpHWIdceavSGrL,遥か彼方,2012-01,j-rock,64.0,False,243.35,0.000024,0.308,0.955,0.113000,E,0.1660,-3.763,Major,0.1110,174.991,0.237
2042,0MK8l3nURwwQIjafvXoJJt,ASIAN KUNG-FU GENERATION,Japan,5ORPYXJKlpHWIdceavSGrL,遥か彼方,2012-01,j-rock,64.0,False,243.35,0.000024,0.308,0.955,0.113000,E,0.1660,-3.763,Major,0.1110,174.991,0.237
2552,0MK8l3nURwwQIjafvXoJJt,ASIAN KUNG-FU GENERATION,Japan,052C0m9kD30nZqcPWPPRqm,Haruka Kanata,2012-01,j-rock,61.0,False,243.35,0.000025,0.316,0.951,0.189000,C#,0.1700,-3.757,Minor,0.1120,174.895,0.255
569,0MK8l3nURwwQIjafvXoJJt,ASIAN KUNG-FU GENERATION,Japan,0WaaPFt4Qy8sVfxKz43bCD,Re:Re:,2004-10,j-rock,59.0,False,228.76,0.000318,0.537,0.864,0.123000,E,0.1080,-5.049,Major,0.0369,153.035,0.727
1059,0MK8l3nURwwQIjafvXoJJt,ASIAN KUNG-FU GENERATION,Japan,052C0m9kD30nZqcPWPPRqm,Haruka Kanata,2012-01,j-rock,59.0,False,243.35,0.000025,0.316,0.951,0.189000,C#,0.1700,-3.757,Minor,0.1120,174.895,0.255
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2266,3PzqP5IkpLhlSdZLh7jwPn,WagakkiBand,Japan,032BHbzqmtBi4P5swU0YFd,星月夜,2015-09,j-rock,33.0,False,258.64,0.094000,0.455,0.978,0.000239,D,0.2730,-1.861,Minor,0.1950,159.987,0.638
1794,3PzqP5IkpLhlSdZLh7jwPn,WagakkiBand,Japan,228F5qKKFgZex7ZinBFlCT,脳漿炸裂ガール,2014-04,j-rock,31.0,False,192.29,0.143000,0.577,0.963,0.000127,C#,0.1870,-2.635,Major,0.1720,155.093,0.814
1583,3PzqP5IkpLhlSdZLh7jwPn,WagakkiBand,Japan,1pkaABwiCDfb8jmSnanJs6,虹色蝶々,2014-04,j-rock,30.0,False,300.25,0.126000,0.587,0.708,0.000092,D,0.3530,-5.535,Minor,0.0270,95.001,0.652
1795,3PzqP5IkpLhlSdZLh7jwPn,WagakkiBand,Japan,7ydjzkY9iUItQ4LWT7FfMH,Valkyrie-戦乙女-,2016-06,j-rock,30.0,False,262.64,0.004210,0.536,0.962,0.000000,G,0.0787,-3.007,Major,0.1100,97.524,0.650


In [136]:
df_top_15.to_csv("../data/cleaned_data/spotify_europeVSjapan_analysis.csv", index=False)